In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import  ChatOpenAI

load_dotenv()

model = ChatOpenAI(
        model_name="qwen3-max",
        api_key=os.getenv("OPENAI_API_KEY"),
        openai_api_base="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

# print(model)

使用 LangGraph 访问大模型的方式

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[],
)

agent.invoke({ "messages": [
    {
        "role": "user",
        "content": "你好"
    }
]})

In [ ]:
for chunk in agent.stream({
    "messages": [
        {
            "role": "user",
            "content": "你是谁，能帮我解决什么问题吗？"
        }
    ],
}, stream_mode="messages"):
    print(chunk)
    print("\n")

 增加工具调用

In [ ]:
import datetime
from langchain.agents import create_agent

def get_current_time():
    """获取当前时间"""
    return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

agent = create_agent(
    model=model,
    tools=[get_current_time],
    system_prompt="你是一个时间助手，能获取当前时间，并返回给用户"
)

agent.invoke({
    "messages": [
        {"role": "user", "content": "当前时间是什么？"}
    ]
})

In [ ]:
from collections.abc import Callable

from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage
from langchain.tools import tool
from langgraph.prebuilt.tool_node import ToolCallRequest


@tool("divide_tool", return_direct=True)
def divide(a: int, b: int) -> float:
    """计算两个整数的除法
    Args:
        a: 被除数
        b: 除数
    """
    if b == 0:
        raise ValueError("除数不能为0")
    return a / b


@wrap_tool_call
def handle_tool_errors(
    request: ToolCallRequest,
    handler: Callable[[ToolCallRequest], ToolMessage],
) -> ToolMessage:
    """把工具异常转成 ToolMessage，让模型能继续处理，而不是直接崩溃。"""
    try:
        return handler(request)
    except Exception as error:
        if isinstance(error, ZeroDivisionError):
            content = "除数不能为0"
        elif isinstance(error, ValueError):
            content = f"输入的参数错误: {error}"
        else:
            content = f"工具执行错误: {error}"
        return ToolMessage(
            content=content,
            tool_call_id=request.tool_call["id"],
        )


# create_agent 会自己创建 ToolNode，不要把 ToolNode 放进 tools=
# 工具错误处理应通过 middleware=[...] 传入
agent_with_handle_tool_error = create_agent(
    model=model,
    tools=[divide],
    middleware=[handle_tool_errors],
    system_prompt="你是一个计算器，能计算两个整数的除法",
)

result = agent_with_handle_tool_error.invoke(
    {
        "messages": [
            {"role": "user", "content": "计算10除以5"},
        ]
    }
)

print(result['messages'][-1].content)

LangGraph 短期记忆

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

checkpointer = InMemorySaver()

def get_weather(city: str) -> str:
    """获取某个城市的天气"""
    return f"城市{city}，天气是晴天！"

agent = create_agent(
    model=model,
    tools=[get_weather],
    checkpointer=checkpointer
)

config = {
    "configurable": {
        "thread_id": 1
    }
}

cs_response = agent.invoke(
    {"messages": [{ "role": "user", "content": "长沙天气怎么样？"}]},
    config
)

print(cs_response["messages"][-1].content)

bj_response = agent.invoke(
    {"messages": [{"role": "user", "content": "北京呢？"}]},
    config
)

print(bj_response["messages"][-1].content)

LangGraph 还提供了状态管理机制，用于保存处理过程中的处理结果。而且这些数据还可以在 Tools 工具中使用

In [ ]:
from typing import Annotated
from langchain.agents import create_agent, AgentState
from langchain.tools import InjectedState, tool


class CustomState(AgentState):
    user_id: str


@tool
def get_user_info(state: Annotated[CustomState, InjectedState]) -> str:
    """获取用户信息"""
    user_id = state["user_id"]
    return f"用户{user_id}，性别男，年龄20"


agent = create_agent(
    model=model,
    tools=[get_user_info],
    state_schema=CustomState,
)

response = agent.invoke(
    {
        "messages": [{"role": "user", "content": "查询用户信息"}],
        "user_id": "cxc",
    }
)

print(response["messages"][-1].content)

Human-in-Loop，人类监督

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


@tool
def book_hotel(name: str) -> str:
    """预订酒店"""
    return f"已预订酒店{name}"


# 不要用 Interrupt(...) 当函数：那是数据结构，不是 suspend API。
# create_agent 做人机确认应使用 HumanInTheLoopMiddleware + checkpointer。
agent = create_agent(
    model=model,
    tools=[book_hotel],
    system_prompt="你是一个酒店预订助手，能预订酒店",
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "book_hotel": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                }
            }
        )
    ],
)

# HITL 必须带 thread_id，才能暂停后再恢复
config = {"configurable": {"thread_id": "hotel-booking-1"}}

result = agent.invoke(
    {"messages": [{"role": "user", "content": "预订北京希尔顿酒店"}]},
    config=config,
)

# 第一次会在工具执行前中断，等待人工决策
interrupts = result.get("__interrupt__")
print("中断信息:", interrupts)

# edit 必须提供 edited_action（改后的工具名和参数），不能只写 type=edit
result = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        "name": "book_hotel",
                        "args": {"name": "北京香格里拉"},
                    },
                }
            ]
        }
    ),
    config=config,
)
print(result["messages"][-1].content)

# 其他决策示例：
# approve: Command(resume={"decisions": [{"type": "approve"}]})
# reject:  Command(resume={"decisions": [{"type": "reject", "message": "用户取消预订"}]})
